# 04 — Evaluate one finished run

Training already wrote every metric and every figure. This notebook reads one run
back and adds the parts that need a person looking at them: the per-cohort
breakdown, the source probe, the confusion structure and the mistakes worth
inspecting.

It trains nothing and it writes nothing into the run folder. It works equally on a
centralised run from notebook 03 and on a federated result collected in notebook
07, because both write the same `predictions_test.csv` layout.

## Configuration

Point `RUN_DIR` at one run folder. The default picks the most recent centralised
run, but the commented lines show the other two things you are likely to want:
a specific run, or one of the thirteen reported federated results.

In [ ]:
from pathlib import Path

# Repository root. The notebook lives in notebooks/, so the parent is the root.
REPO_ROOT = Path.cwd().parent

# Where notebook 03 writes its runs. Expect one test_NNN_* folder per run.
CLASSIFIER_RESULTS = REPO_ROOT / "results" / "classifier"

# Where the thirteen reported experiments live, one folder per test.
FEDERATED_RESULTS = REPO_ROOT / "results" / "federated"

# Which run to read. It must contain results.json and predictions_test.csv.
_candidates = sorted(d for d in CLASSIFIER_RESULTS.glob("test_*")
                     if (d / "results.json").is_file())
RUN_DIR = _candidates[-1] if _candidates else None

# RUN_DIR = CLASSIFIER_RESULTS / "test_002_resnet18_subtype"          # a specific run
# RUN_DIR = FEDERATED_RESULTS / "test01_centralized" / "seed_42"      # the reported baseline
# RUN_DIR = FEDERATED_RESULTS / "test09_fedprox_skewed"               # best federated run

# The margin below which a difference is not a result. Measured on this task
# between two runs of a byte-identical configuration differing only in seed.
NOISE_FLOOR = 0.067

assert RUN_DIR is not None and RUN_DIR.is_dir(), (
    f"no run found. Train one with notebook 03, or set RUN_DIR by hand.")
print(f"reading {RUN_DIR}")
for f in sorted(RUN_DIR.iterdir()):
    if f.is_file():
        print(f"  {f.name}")

## Imports

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

plt.rcParams.update({"figure.dpi": 120, "font.size": 9, "axes.grid": True,
                     "grid.alpha": 0.25, "axes.spines.top": False,
                     "axes.spines.right": False})

results = json.loads((RUN_DIR / "results.json").read_text())
config = results.get("config", {})
CLASS_NAMES = config.get("class_names", ["HRposHER2neg", "TripleNeg", "HER2pos"])
PROB_COLS = [f"prob_{n}" for n in CLASS_NAMES]

print(f"model  {config.get('model', config.get('model_name', '?'))}")
print(f"seed   {config.get('seed')}")
print(f"epochs {results.get('epochs_run')}, best at {results.get('best_epoch')}")

## The headline numbers

Accuracy on its own says nothing here. It is printed beside the trivial baseline,
which is the accuracy of always predicting the majority class, because that is the
only way to see whether the model does anything a constant predictor does not.

The generalisation gap is the training accuracy at the selected epoch minus the
test accuracy. A large gap with a decent AUC means the model memorised the training
patients and still generalised a little.

In [ ]:
splits = results.get("splits", {})
rows = []
for split, m in splits.items():
    rows.append({
        "split": split,
        "n": m.get("n_patients"),
        "macro_auc": round(m["auc"], 4),
        "accuracy": round(m["accuracy"], 4),
        "trivial": round(m["trivial_baseline_accuracy"], 4),
        "above_trivial": round(m["accuracy"] - m["trivial_baseline_accuracy"], 4),
        "balanced_acc": round(m["balanced_accuracy"], 4),
        "macro_f1": round(m["macro_f1"], 4),
    })
print(pd.DataFrame(rows).to_string(index=False))

gap = results.get("generalisation_gap", {})
if gap:
    print(f"\ntrain accuracy at the selected epoch: "
          f"{results.get('train_acc_at_best', float('nan')):.4f}")
    for split, g in gap.items():
        print(f"  gap against {split}: {g:+.4f}")

print(f"\nnoise floor {NOISE_FLOOR} macro-AUC — nothing smaller than this is a difference")

## Per class

Macro AUC averages the three classes equally, which hides the fact that they are
not equally learnable here. HER2+ is the smallest class and consistently the
hardest, and a run can look respectable overall while barely finding it at all.

Read the recall column against the class count. A recall of 0.19 on 53 patients
means the model found 10 of them.

In [ ]:
t = splits["test"]
per_class = pd.DataFrame({
    "class": CLASS_NAMES,
    "n": t["class_counts"],
    "auc": t["per_class_auc"],
    "precision": t["per_class_precision"],
    "recall": t["per_class_recall"],
    "f1": t["per_class_f1"],
})
per_class["found"] = [int(round(r * n)) for r, n in
                      zip(t["per_class_recall"], t["class_counts"])]
print(per_class.to_string(index=False))

## The confusion matrix

Two runs with the same macro AUC can predict entirely different classes, and
without this that stays invisible. Rows are the truth, columns the prediction.

The row-normalised panel is the one to read for class behaviour. A row that is
almost entirely in one column means that class is being absorbed into another.

In [ ]:
cm = np.array(t["confusion"], dtype=float)
pct = cm / np.clip(cm.sum(axis=1, keepdims=True), 1e-9, None)

fig, axes = plt.subplots(1, 2, figsize=(4.2 * len(CLASS_NAMES), 4.2))
for ax, data, fmt, lab in ((axes[0], cm, "{:.0f}", "count"),
                           (axes[1], pct, "{:.2f}", "row-normalised")):
    im = ax.imshow(data, cmap="Blues", vmin=0)
    ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(f"Test — {lab}", fontsize=9); ax.grid(False)
    thr = data.max() / 2 if data.max() else 0.5
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            ax.text(j, i, fmt.format(data[i, j]), ha="center", va="center",
                    color="white" if data[i, j] > thr else "black", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

## Per cohort

Only meaningful when the run pooled cohorts, which the reported experiments do.

This matters because a single overall number hides a large spread. The dataset
authors report AUC 0.78 on I-SPY2 against 0.54 on DUKE for one model, which is a
0.24 range inside one result.

The AUC is recomputed here from the saved per-patient probabilities rather than
read from a stored field, so it is a genuine independent check of the headline
number when you group by nothing.

In [ ]:
pred = pd.read_csv(RUN_DIR / "predictions_test.csv")

def macro_auc(frame):
    """Macro one-vs-rest AUC from the saved per-patient probabilities."""
    p = frame[PROB_COLS].to_numpy()
    if frame.label.nunique() < len(CLASS_NAMES):
        return float("nan")
    if len(CLASS_NAMES) == 2:
        return float(roc_auc_score(frame.label, p[:, 1]))
    return float(roc_auc_score(frame.label, p, multi_class="ovr", average="macro"))

# Sanity check: recomputing over everything must reproduce the stored value.
print(f"recomputed overall macro AUC {macro_auc(pred):.10f}")
print(f"stored in results.json       {t['auc']:.10f}")

if "cohort" in pred.columns and pred.cohort.nunique() > 1:
    rows = []
    for c, g in pred.groupby("cohort"):
        rows.append({"cohort": c, "n": len(g), "macro_auc": round(macro_auc(g), 4),
                     "accuracy": round(g.correct.mean(), 4),
                     "majority": round(g.label_name.value_counts().max() / len(g), 4)})
    breakdown = pd.DataFrame(rows).sort_values("n", ascending=False)
    print()
    print(breakdown.to_string(index=False))
    spread = breakdown.macro_auc.max() - breakdown.macro_auc.min()
    print(f"\nspread across cohorts: {spread:.4f}"
          f"{'  — wider than the noise floor' if spread > NOISE_FLOOR else ''}")
else:
    print("\nsingle cohort — no breakdown to make")

## The source probe

The check that has to run before any pooled result is believed. It trains the
identical pipeline with the cohort as the label instead of the subtype. If the
images identify their own cohort, then a model can reach a respectable subtype
score by learning the cohort and its class prior, with no biology involved.

| probe macro AUC | what it means |
|---|---|
| 0.90 or above | the images identify the cohort trivially, the result is contaminated |
| around 0.70 | a signature exists but does not dominate, report it beside the result |
| around 0.50 | the cohorts are indistinguishable, pooling is safe |

Measured on this pooled three-cohort dataset: **0.9978**, against 0.6068 for the
actual subtype task.

That is not a reason to abandon the dataset. It is a reason to state the confound
beside every pooled number, and it is exactly what makes the federated hospitals in
notebook 06 genuinely non-IID rather than merely different in size.

The probe is a full training run, so the cell below does not launch it by default.

In [ ]:
# Running the probe means training a whole model with the cohort as the label.
# It is a real GPU cost, so it is off by default. To run it, open notebook 03 and
# change the label column in the config cell:
#
#   TASK_COLUMN, CLASS_NAMES = "dataset", ("spy2", "spy1", "duke")
#
# then rebuild the dataset CSVs with notebook 02 and train as usual. The recorded
# result is below.

PROBE_RESULT = 0.9978          # macro AUC predicting the COHORT, pooled dataset
SUBTYPE_RESULT = 0.6068        # macro AUC predicting the SUBTYPE, same pipeline

print(f"source probe (cohort as label)  {PROBE_RESULT:.4f}")
print(f"the actual task (subtype)       {SUBTYPE_RESULT:.4f}")
print(f"difference                      {PROBE_RESULT - SUBTYPE_RESULT:+.4f}")
print()
print("The images identify their cohort almost perfectly. DUKE is 64.6% HRposHER2neg")
print("against I-SPY2's 38.8%, and its tumours are about five times smaller by volume,")
print("so 'small tumour, so DUKE, so HRposHER2neg' is available as a shortcut.")

## Where it went wrong

The mistakes ranked by how confident the model was in them. A confident mistake is
more informative than an uncertain one: it points at a systematic failure rather
than at a borderline case.

In [ ]:
wrong = pred[~pred.correct].copy()
print(f"{len(wrong)} of {len(pred)} patients misclassified\n")
print(pd.crosstab(pred.label_name, pred.pred_name, margins=True).to_string())

wrong["confidence"] = wrong[PROB_COLS].max(axis=1)
cols = ["pid", "label_name", "pred_name", "confidence"]
if "cohort" in wrong.columns:
    cols.insert(1, "cohort")
print("\nmost confident mistakes:")
print(wrong.nlargest(10, "confidence")[cols].round(4).to_string(index=False))

print("\nleast confident predictions overall (the model knows it does not know):")
pred_conf = pred.assign(confidence=pred[PROB_COLS].max(axis=1))
print(pred_conf.nsmallest(5, "confidence")[
    [c for c in cols if c != "confidence"] + ["confidence", "correct"]
].round(4).to_string(index=False))

## The training curves

Already written as PNGs by notebook 03. Redrawn here from `train_log.csv` so they
can be zoomed and read interactively, which is the whole reason to open a run in a
notebook rather than a file browser.

The gap panel is the one to watch. Training accuracy climbing while validation
accuracy sits flat is memorisation, and the epoch where they separate is the epoch
after which more training buys nothing.

In [ ]:
log_path = RUN_DIR / "train_log.csv"
if not log_path.is_file():
    print("no train_log.csv — this is a collected federated result, not a local run")
else:
    hist = pd.read_csv(log_path)
    best = results.get("best_epoch")

    fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
    ax[0].plot(hist.epoch, hist.train_loss, label="train", color="#4c78a8")
    ax[0].plot(hist.epoch, hist.val_loss, label="validation", color="#e45756")
    ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend(fontsize=7)

    ax[1].plot(hist.epoch, hist.val_auc, color="#e45756")
    ax[1].axhline(0.5, color="grey", ls=":", lw=1)
    ax[1].set_title("Validation macro-AUC (patient level)"); ax[1].set_xlabel("epoch")

    ax[2].plot(hist.epoch, hist.train_acc - hist.val_accuracy, color="#f58518")
    ax[2].axhline(0, color="grey", ls=":", lw=1)
    ax[2].set_title("Generalisation gap (train − val accuracy)")
    ax[2].set_xlabel("epoch")

    for a in ax:
        if best:
            a.axvline(best, color="k", ls="--", lw=1, alpha=0.6)
    plt.tight_layout(); plt.show()

    print(f"selected epoch {best} of {len(hist)}")
    print(f"validation AUC at that epoch: {hist.loc[hist.epoch == best, 'val_auc'].iloc[0]:.4f}")
    print(f"30-epoch mean {hist.val_auc.mean():.4f}, sd {hist.val_auc.std():.4f}")
    z = (hist.loc[hist.epoch == best, "val_auc"].iloc[0] - hist.val_auc.mean()) / hist.val_auc.std()
    print(f"the selected epoch sits {z:.2f} standard deviations above its own mean")
    print("\nA selected epoch far above its own mean is a warning: the checkpoint was")
    print("chosen to maximise exactly the quantity it is then reported on.")

## What this notebook produced

Nothing on disk. It is a read-only view of one run, on purpose, so that opening a
result can never modify it.

**What to carry away from it**

The headline table, the per-class recall against the class counts, the per-cohort
spread, and how far above its own mean the selected epoch sat. That last one is the
check that caught a federated run selecting its global model at round 2 of 30 and
reporting inflated per-hospital numbers as a result.

**Where this goes next**

Notebook 05 puts several runs side by side with the noise floor applied.